# Analyze button — crew pipeline

This notebook explains what happens when the user clicks **Analyze**: Gradio
collects inputs, the pipeline builds a six-agent CrewAI crew with MCP tools,
and a typed analysis package is produced (or a deterministic fallback).

**Source files**
- `app.py` — `analyze`
- `careercrew/pipeline.py` — `run_analysis`
- `careercrew/crew.py` — `build_crew`
- `config/agents.yaml`, `config/tasks.yaml`
- Detail doc: `docs/CODE_PATH_ANALYZE.md`


## Overview

Analyze is the core of CareerCrew. Gradio gathers the resume and JD (upload or
paste), runs input guardrails, then `pipeline.run_analysis` builds a CrewAI crew
of six agents with MCP tools.

The crew produces a typed package: profile, requirements, gaps, rewrites, prep
plan, and screening answers. If the LLM crew fails, CareerCrew still returns a
**deterministic fallback package** so the demo keeps working. The result is
submitted into HITL review and rendered across the UI tabs.


## Call chain

```
Gradio Analyze
   app.analyze
      resolve uploads / paste
      guardrails.validate_request
      pipeline.run_analysis
         build_crew (agents.yaml + tasks.yaml + MCP)
         crew.kickoff
         parse → Pydantic models
         OR deterministic_package (fallback)
      review.submit_for_review
      session_store persist
   render Match / Prep / Screening tabs
```


## Agents and tools

| Piece | Location |
|-------|----------|
| Agent roles | `config/agents.yaml` |
| Task prompts | `config/tasks.yaml` |
| Crew wiring | `careercrew/crew.py` → `build_crew` |
| Shared tool logic | `careercrew/document_tools.py` |
| MCP server | `mcp_server/server.py` — `read_resume`, `read_job_description`, `query_answer_bank` |
| Contracts | `careercrew/models.py` |
| Honesty | `data/answer_bank.yaml` (`verified: true` only) |


In [ ]:
# Six-step Analyze crew sequence

STEPS = [
    ("1 Resume reader", "Structured CandidateProfile"),
    ("2 JD reader", "JobRequirements"),
    ("3 Gap analyst", "Coverage score + GapReport"),
    ("4 Rewrite coach", "Cited resume rewrites"),
    ("5 Prep advisor", "InterviewPrepPlan (seniority-aware)"),
    ("6 Screening drafter", "Answers ONLY from verified answer bank"),
]

print("Analyze crew sequence")
for name, out in STEPS:
    print(f"  {name:20} -> {out}")

print()
print("Fallback:")
print("  crew fails / no key -> pipeline.deterministic_package(...)")
print("  still grounded in gaps + answer bank retrieval")


## Guardrails

- **Input:** file type/size, one resume source, one JD source, no raw job-board scrape.
- **Tools:** upload confinement; answer bank returns only verified facts.
- **Output:** Pydantic validation; prep advice stays grounded; screening never invents unverified claims.

Implementation: `careercrew/guardrails.py`.


## Code reference

| Step | File | Symbol |
|------|------|--------|
| UI | `app.py` | `analyze` |
| Orchestrator | `pipeline.py` | `run_analysis`, `deterministic_package` |
| Crew | `crew.py` | `build_crew` |
| MCP | `mcp_server/server.py` | three tools |
| HITL handoff | `review.py` | `submit_for_review` |

Continue with `04_hitl_and_coaches.ipynb`.
